In [3]:
# -*-Encoding: utf-8 -*-
import os
import sys
sys.path.insert(0, os.path.abspath('../'))
from data_load.data_loader import Dataset_Custom
from model.model import diffusion_generate, denoise_net, pred_net
from model.jiucai_model import jiucai_net
from torch.optim.lr_scheduler import OneCycleLR, StepLR

from gluonts.torch.util import copy_parameters
from utils.tools import EarlyStopping, adjust_learning_rate
from model.resnet import Res12_Quadratic
from model.diffusion_process import GaussianDiffusion

from model.encoder import Encoder
from model.embedding import DataEmbedding
import numpy as np
import math
import collections
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import DataLoader
import torch.nn.functional as F

import os
import time
import warnings
import pandas as pd
from utils.timefeatures import time_features
import ast

In [4]:
import argparse
parser = argparse.ArgumentParser(description='generating')

# Load data
parser.add_argument('--root_path', type=str, default='./data/2016', help='root path of the data files')
parser.add_argument('--checkpoints', type=str, default='./checkpoints/', help='location of model checkpoints')
parser.add_argument('--sequence_length', type=int, default=10, help='length of input sequence')
parser.add_argument('--prediction_length', type=int, default=10, help='prediction sequence length')
parser.add_argument('--target_dim', type=int, default=1, help='dimension of target')
parser.add_argument('--input_dim', type=int, default=24, help='dimension of input')
parser.add_argument('--hidden_size', type=int, default=128, help='encoder dimension')
parser.add_argument('--embedding_dimension', type=int, default=64, help='feature embedding dimension')

# Diffusion process
parser.add_argument('--diff_steps', type=int, default=1000, help='number of the diff step')
parser.add_argument('--dropout_rate', type=float, default=0.1, help='dropout')
parser.add_argument('--beta_schedule', type=str, default='linear', help='the schedule of beta')
parser.add_argument('--beta_start', type=float, default=0.0, help='start of the beta')
parser.add_argument('--beta_end', type=float, default=1.0, help='end of the beta')
parser.add_argument('--scale', type=float, default=0.1, help='adjust diffusion scale')

# Bidirectional VAE
parser.add_argument('--arch_instance', type=str, default='res_mbconv', help='path to the architecture instance')
parser.add_argument('--mult', type=float, default=1, help='mult of channels')
parser.add_argument('--num_layers', type=int, default=2, help='num of RNN layers')
parser.add_argument('--num_channels_enc', type=int, default=32, help='number of channels in encoder')
parser.add_argument('--channel_mult', type=int, default=2, help='number of channels in encoder')
parser.add_argument('--num_preprocess_blocks', type=int, default=1, help='number of preprocessing blocks')
parser.add_argument('--num_preprocess_cells', type=int, default=3, help='number of cells per block')
parser.add_argument('--groups_per_scale', type=int, default=2, help='number of cells per block')
parser.add_argument('--num_postprocess_blocks', type=int, default=1, help='number of postprocessing blocks')
parser.add_argument('--num_postprocess_cells', type=int, default=2, help='number of cells per block')
parser.add_argument('--num_channels_dec', type=int, default=32, help='number of channels in decoder')
parser.add_argument('--num_latent_per_group', type=int, default=8, help='number of channels in latent variables per group')

# Training settings
parser.add_argument('--num_workers', type=int, default=1, help='data loader num workers') #set to 1 for debug
parser.add_argument('--patience', type=int, default=3, help='early stopping patience')
parser.add_argument('--itr', type=int, default=5, help='experiment times')
parser.add_argument('--train_epochs', type=int, default=20, help='train epochs')
parser.add_argument('--batch_size', type=int, default=16, help='batch size of train input data')
parser.add_argument('--learning_rate', type=float, default=0.0005, help='optimizer learning rate')
parser.add_argument('--weight_decay', type=float, default=0.0000, help='weight decay')
parser.add_argument('--zeta', type=float, default=0.5, help='trade off parameter zeta')
parser.add_argument('--eta', type=float, default=1.0, help='trade off parameter eta')

parser.add_argument('--use_attn', type=bool, default=True, help='use attention to do cross conditional control')
parser.add_argument('--embed_dim', type=int, default=22, help='vetors dim length')
parser.add_argument('--num_heads', type=int, default=2, help='attention head numbers')


_StoreAction(option_strings=['--num_heads'], dest='num_heads', nargs=None, const=None, default=2, type=<class 'int'>, choices=None, required=False, help='attention head numbers', metavar=None)

In [5]:
args = parser.parse_args([])

In [6]:
df_test = pd.read_csv('/home/userroot/projs/xiaojiucai/astock/data/test.csv',sep='\t')
df_train = pd.read_csv('/home/userroot/projs/xiaojiucai/astock/data/train.csv',sep='\t')

In [7]:
stock = df_train[['CODE', 'DATE', 'text_a', 'stock_factors']]
sorted_stock = stock.sort_values(['CODE', 'DATE'])
sorted_stock = sorted_stock.reset_index(drop=True)

In [8]:
df_stamp = pd.DatetimeIndex(sorted_stock['DATE']) #训练数据集的日期标签
data_stamp = time_features(df_stamp) #把日期时间序列转化为多个时间层次的序列，并且把序列的数值归一到【-0.5， +0.5】

# output, y_noisy, dsm_loss = self.denoise_net(batch_x, x_mark, batch_y, t)

In [9]:
sorted_stock['stock_factors'] = sorted_stock['stock_factors'].apply(ast.literal_eval)

In [10]:
xx = []
X = None
cnt = 0
for g in sorted_stock.groupby('CODE'):
    #print(len(g[1]))
    
    
    if len(g[1]) > 10:
        tmp = g[1][['stock_factors']].iloc[0:args.sequence_length].apply(np.array)
        x = torch.tensor(np.array(tmp.values.tolist()), dtype=torch.float32)
        x = x.squeeze()
        #print(x.shape)
        xx.append(x)
        cnt += 1
        #break
    if cnt == args.batch_size:
        X = torch.stack(xx)
        break

print(X.shape)

torch.Size([16, 10, 24])


In [11]:
tmp = sorted_stock[['stock_factors']].iloc[0:args.sequence_length].apply(np.array)
x_input = X.to('cuda')
y = x_input
t = torch.tensor(data_stamp[0:args.sequence_length])
t = t.unsqueeze(0).to('cuda')
diff_t = torch.randint(0, args.diff_steps, (args.batch_size,)).long().to('cuda')

In [12]:
x_input.shape

torch.Size([16, 10, 24])

In [13]:
%load_ext autoreload

%autoreload 2
%reload_ext autoreload
from model.jiucai_model import jiucai_net
#import importlib
#importlib.reload(jiucai_net)
d_net = denoise_net(args).to('cuda')
j_net = jiucai_net(args).to('cuda')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
%reload_ext autoreload
j_net(x_input,t,y,diff_t)

attention doing


(<model.encoder.NormalDecoder at 0x7f16df6da320>,
 tensor([[[[-2.8550, -2.3735,  0.1076,  ...,  0.0939, -0.1597, -0.8456],
           [-0.0852,  0.5501,  0.7444,  ...,  1.0962,  1.3046,  1.0550],
           [ 1.1990, -0.5805,  1.4335,  ..., -0.3766,  0.4283, -0.8325],
           ...,
           [ 0.3398,  0.3893,  0.1961,  ...,  0.7910, -0.7925,  0.2966],
           [ 0.8851, -0.2641,  0.5173,  ..., -1.7034, -0.2819,  1.5973],
           [ 0.4576,  1.0565,  0.6780,  ...,  0.8202,  0.3461,  1.9076]]],
 
 
         [[[ 0.2337,  0.1474,  0.4226,  ...,  0.4816, -0.7482,  0.5633],
           [ 0.7558,  1.2892,  1.4534,  ...,  0.4314, -0.9026, -0.4034],
           [-1.0666, -0.9324, -0.2303,  ..., -0.2726,  1.2091,  1.0102],
           ...,
           [ 0.0782,  0.7620, -0.8491,  ...,  0.5616, -0.9969,  0.9088],
           [-1.2032,  0.4600, -1.0604,  ...,  0.9862,  0.0223, -0.4843],
           [-1.1894,  1.2325,  0.7226,  ...,  0.1711, -2.6913,  0.5823]]],
 
 
         [[[ 0.1443,  0.4280, 

In [18]:
d_net(x_input,t,y,diff_t)

(<model.encoder.NormalDecoder at 0x7f107c1974f0>,
 tensor([[[[ 0.3403,  2.6290, -0.5521,  ..., -0.4688,  0.5646,  1.3136],
           [ 0.1651, -0.6179,  0.7537,  ...,  0.6219,  1.0547, -0.5237],
           [-0.2875,  0.8194, -0.8593,  ...,  0.0109, -0.0260, -2.3500],
           ...,
           [-0.5739,  0.3590,  1.1618,  ...,  0.0195, -0.6214,  2.0949],
           [ 1.4808,  0.1285,  0.3615,  ..., -2.1650, -1.3665,  0.2000],
           [-0.0446,  0.4247, -1.7133,  ..., -0.5341,  0.6213, -0.3703]]],
 
 
         [[[ 0.2550,  0.1486,  0.6640,  ..., -0.7608, -0.0219,  0.4251],
           [-0.9666,  0.3326, -0.3552,  ..., -1.1109,  1.6931, -0.1372],
           [ 0.9941,  0.5534,  0.2706,  ..., -0.7913, -0.4540, -1.0752],
           ...,
           [-0.1274,  0.8503, -0.7400,  ..., -0.9243, -0.4481,  1.7436],
           [-0.2217, -0.2210, -1.0468,  ...,  0.4021,  0.7329, -0.7587],
           [ 0.7906,  0.3310, -0.9997,  ...,  0.4705,  0.3664,  0.6591]]],
 
 
         [[[ 0.1609, -0.4808, 